In [9]:
import chromadb
import openai
from chromadb.config import Settings
from chromadb import PersistentClient
import pandas as pd

client = PersistentClient(path="./chroma_db")

In [19]:
#client.delete_collection("looted_text")
collection = client.create_collection(
    name="looted_text",
    configuration={
        "hnsw": {
            "space": "cosine",       # recommandé pour du texte
            "ef_construction": 200,
            "ef_search": 150
        }
    }
)

In [20]:
df = pd.read_csv("../data/clean/artifacts_comparison.csv", sep = ';', encoding="utf-8-sig")

In [23]:
# Ajoutez les documents des textes web
collection.add(
    ids=[f"b_{i}" for i in range(len(df))] + [f"c_{i}" for i in range(len(df))],
    documents=df["wb_txt_shrt"].tolist() + df["wb_txt_lg"].tolist(),
    metadatas=[{"source": "wb_txt_shrt", "artifact_id": str(v)} for v in df["artifact_id"]] +
              [{"source": "wb_txt_lg", "artifact_id": str(v)} for v in df["artifact_id"]],
)

# results for each
res_B = collection.query(
    query_texts=df["looted_text"].tolist(),
    n_results=1,
    where={"source": "wb_txt_shrt"},
    include=["distances", "metadatas"]
)

res_C = collection.query(
    query_texts=df["looted_text"].tolist(),
    n_results=1,
    where={"source": "wb_txt_lg"},
    include=["distances", "metadatas"]
)

# viz df
df["wb_txt_shrt_dist"] = [d[0] for d in res_B["distances"]]
df["wb_txt_lgidst"] = [d[0] for d in res_C["distances"]]


C:\Users\Utilisateur\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 8.92MiB/s]


In [26]:
df.to_csv("first_text_scoring.csv",sep = ';', encoding="utf-8-sig")

In [2]:
import os
a = "data_cleaning.ipynb" # Relative path
b = os.path.abspath(a)    # Get absolute path

print(b)

C:\Users\Utilisateur\Desktop\IRONHACK_DA\COURSES\FINAL_PROJECT\final_project_looted_artefacts\notebooks\data_cleaning.ipynb
